# Самостоятельная работа: RGB DCGAN для лиц Olivetti Faces

В этой работе нужно самостоятельно собрать и исследовать DCGAN для
генерации лиц. Olivetti Faces отличается от LFW: это небольшой набор
из 400 чёрно-белых лиц размером $64\times64$. Канал повторяется три
раза только для совместимости с RGB-архитектурой; новых цветовых
сведений такое преобразование не создаёт.

Все заголовки описывают этапы. Маркеры `TODO(student)` находятся
исключительно в коде и обозначают место самостоятельной реализации.

## Короткий сюжет опыта

Вы строите художника, который рисует лицо по карточке из 100 чисел, и
проверяющего, который отличает рисунок от фотографии. За одну итерацию
сначала учится проверяющий, а затем художник получает его замечания.
Каждая следующая ячейка создаёт одну нужную часть этой игры. Не
переходите дальше, пока контрольная точка текущего этапа не выполнена.

### Словарь перед первой реализацией

Параметры — числа внутри слоёв, которые можно менять. Прямой проход
создаёт картинку и ответ проверяющего. Потеря — одно число, описывающее
ошибку ответа. Градиент — подсказка, как немного изменить параметры.
`GradientTape` находит такие подсказки, а Adam применяет их к выбранной
сети.

## Этап 1. Конфигурация и назначение данных

**Цель:** определить воспроизводимый эксперимент. Доступны все
константы ниже. Метки субъектов Olivetti нельзя передавать в GAN:
генератор не классифицирует людей, а дискриминатор видит только
реальное или синтетическое изображение.

**Контрольная точка:** реальный пакет должен иметь форму
$(B,64,64,3)$, а шум — $(B,100)$.

`(B,64,64,3)` означает «B изображений, у каждого 64 строки, 64
столбца и три канала». Три одинаковых канала здесь нужны не для
создания цвета, а чтобы форма совпадала с учебной RGB-архитектурой.
Метка субъекта — подпись для человека, но не ответ для GAN.

In [ ]:
# get_ipython нужен только для Jupyter: эта команда заставляет
# Matplotlib рисовать внутри ноутбука, без отдельного окна Figure.
from __future__ import annotations

from pathlib import Path
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.datasets import fetch_olivetti_faces

# Сначала зафиксируйте конфигурацию, затем seed — до загрузки и
# создания модели, чтобы срез данных, веса и шум можно было повторить.
SEED = 42
MAX_EXAMPLES = 400
IMAGE_SIZE = 64
CHANNELS = 3
NOISE_DIM = 100
BATCH_SIZE = 32
EPOCHS = 120
LEARNING_RATE = 2e-4
BETA_1 = 0.5
REAL_LABEL = 0.9
# В ARTIFACT_DIR сохраняются только изображения/графики эксперимента.
ARTIFACT_DIR = Path('artifacts/olivetti_face_gan_tensorflow')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)

## Этап 2. Получение Olivetti Faces и переход к RGB-форме

**Требуется:** загрузить набор с фиксированным `random_state`, взять
изображения формы $(N,64,64)$, добавить последнюю ось и повторить её
три раза. Затем преобразовать диапазон $[0,1]$ в $[-1,1]$ и создать
`tf.data.Dataset` из изображений без идентификаторов людей.

**Проверка:** `x_real.shape == (MAX_EXAMPLES, 64, 64, 3)`; все
значения конечны и принадлежат $[-1,1]$.

**Сначала → затем → проверить.** Сначала получите серое лицо в
`[0,1]`. Затем повторите канал и получите RGB-форму. После этого
переведите числа в `[-1,1]`: именно в этой шкале будет рисовать `tanh`
генератора. Наконец сформируйте `dataset` только из `x_real`; ID людей
в него не кладётся. Покажите одну картинку до обучения: это быстро
обнаружит перепутанную шкалу или оси.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

# ----------------------------------------------------------------------
# Конфигурация (фиксированный random_state для воспроизводимости)
# ----------------------------------------------------------------------
SEED = 42
MAX_EXAMPLES = 400          # Olivetti Faces: всего 400 изображений
IMAGE_SIZE = 64             # 64×64
NUM_CHANNELS = 3            # после повтора серого канала
BATCH_SIZE = 32             # меньше батч — датасет крошечный

tf.keras.utils.set_random_seed(SEED)
artifact_dir = Path('artifacts/olivetti_faces_dense_gan')
artifact_dir.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------------
# Шаг 1. Загрузить Olivetti Faces с фиксированным random_state.
#         fetch_olivetti_faces даёт images (N,64,64) в [0,1] и
#         subject_ids (N,). subject_ids оставляем для подписей,
#         но в Dataset его НЕ кладём.
# ----------------------------------------------------------------------
from sklearn.datasets import fetch_olivetti_faces

faces = fetch_olivetti_faces(shuffle=True, random_state=SEED)
x_gray = faces.images[:MAX_EXAMPLES]          # (N, 64, 64), float32 в [0,1]
subject_ids = faces.target[:MAX_EXAMPLES]     # (N,) — только для подписи

assert x_gray.shape == (MAX_EXAMPLES, IMAGE_SIZE, IMAGE_SIZE), x_gray.shape
assert x_gray.min() >= 0.0 and x_gray.max() <= 1.0, \
    f'диапазон не [0,1]: [{x_gray.min()}, {x_gray.max()}]'


# ----------------------------------------------------------------------
# Шаг 2. «Затем»: добавить последнюю ось и повторить серый канал 3 раза.
#         Пока это всё ещё [0,1]; шкалу меняем СЛЕДУЮЩИМ шагом.
# ----------------------------------------------------------------------
x_real = np.repeat(x_gray[..., np.newaxis], NUM_CHANNELS, axis=-1)  # (N,64,64,3)

assert x_real.shape == (MAX_EXAMPLES, IMAGE_SIZE, IMAGE_SIZE, NUM_CHANNELS)
assert x_real.dtype == np.float32 or x_real.dtype == np.float64


# ----------------------------------------------------------------------
# Шаг 3. «Затем»: линейно [0,1] -> [-1,1]. Именно в этой шкале
#         рисует tanh генератора.
# ----------------------------------------------------------------------
x_real = (x_real * 2.0) - 1.0                 # [0,1] -> [-1,1]

assert np.isfinite(x_real).all(), 'non-finite in x_real'
assert x_real.min() >= -1.0 and x_real.max() <= 1.0, \
    f'вне [-1,1]: [{x_real.min()}, {x_real.max()}]'


# ----------------------------------------------------------------------
# Шаг 4. Dataset ТОЛЬКО из изображений. subject_ids в него не кладём.
#         До batching — (64,64,3), после — (B,64,64,3).
# ----------------------------------------------------------------------
dataset = (
    tf.data.Dataset.from_tensor_slices(x_real)
    .shuffle(len(x_real), seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)


# ----------------------------------------------------------------------
# Шаг 5. «Проверить»: форма, конечность, диапазон, первый пакет.
#         Показать одну картинку ДО обучения — так видно перепутанную
#         шкалу или оси.
# ----------------------------------------------------------------------
if __name__ == '__main__':
    print(f'x_gray: shape={x_gray.shape}, range=[{x_gray.min():.3f}, {x_gray.max():.3f}]')
    print(f'x_real: shape={x_real.shape}, range=[{x_real.min():.3f}, {x_real.max():.3f}]')
    print(f'subject_ids: shape={subject_ids.shape} (в Dataset НЕ передаётся)')

    # проверка элемента до batching и первого пакета
    first_elem = next(iter(tf.data.Dataset.from_tensor_slices(x_real).take(1)))
    assert first_elem.shape == (IMAGE_SIZE, IMAGE_SIZE, NUM_CHANNELS), first_elem.shape

    for batch in dataset.take(1):
        assert batch.shape == (BATCH_SIZE, IMAGE_SIZE, IMAGE_SIZE, NUM_CHANNELS), batch.shape
        b = batch.numpy()
        assert np.isfinite(b).all()
        assert b.min() >= -1.0 and b.max() <= 1.0
        print(f'элемент до batching: {first_elem.shape}  (64,64,3)')
        print(f'первый пакет:       {batch.shape}  (B,64,64,3)')

    # показать одну картинку до обучения
    img = (x_real[0] + 1.0) / 2.0              # обратно в [0,1] для imshow
    fig, ax = plt.subplots(figsize=(3, 3))
    ax.imshow(img, vmin=0, vmax=1)
    ax.set_title(f'subject_id={subject_ids[0]} (только подпись)')
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(artifact_dir / 'one_face.png', dpi=120)
    plt.close(fig)
    print(f'сохранено: {artifact_dir / "one_face.png"}')

## Этап 3. Свёрточный генератор

**Требуется:** реализовать генератор, который преобразует
$(B,100)$ в $(B,64,64,3)$: `Dense(4·4·512)`, reshape и четыре
`Conv2DTranspose` с промежуточными BatchNormalization и ReLU. На
последнем слое используйте `tanh`.

**Контрольная точка:** диагностический проход выдаёт конечные значения
в диапазоне $[-1,1]$, а не плоский вектор пикселей.

Генератор сначала получает одну карточку из 100 чисел на лицо. Он
делает маленькую внутреннюю картинку, а затем несколько раз увеличивает
её. Транспонированная свёртка — не волшебное увеличение готовой
фотографии: она применяет обучаемые правила и создаёт новые пиксели.
`BatchNormalization` помогает числам внутри сети оставаться удобного
масштаба, а `tanh` ограничивает финальные RGB-значения.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import tensorflow as tf

# ----------------------------------------------------------------------
# Конфигурация
# ----------------------------------------------------------------------
SEED = 42
NOISE_DIM = 100
IMAGE_HEIGHT = 64
IMAGE_WIDTH = 64
NUM_CHANNELS = 3
BATCH_SIZE = 32

tf.keras.utils.set_random_seed(SEED)
artifact_dir = Path('artifacts/olivetti_faces_dense_gan')
artifact_dir.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------------
# Шаг 1. Фабрика генератора. Создаёт независимый θ_G, НЕ обучает.
# ----------------------------------------------------------------------
def make_generator(noise_dim: int = NOISE_DIM) -> tf.keras.Model:
    """DCGAN G: (B,100) -> (B,64,64,3), выход в [-1,1] через tanh."""
    init = tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02, seed=SEED)

    # 4×4×512 = 8192 — «маленькая внутренняя картинка» после Dense
    start_channels = 512
    start_spatial = 4
    flat_dim = start_spatial * start_spatial * start_channels  # 8192

    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(noise_dim,)),                    # (B,100)

            # внутренняя карта: 100 -> 8192, затем reshape в 4×4×512
            tf.keras.layers.Dense(flat_dim, use_bias=False, kernel_initializer=init),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),
            tf.keras.layers.Reshape((start_spatial, start_spatial, start_channels)),

            # 4×4×512 -> 8×8×256
            tf.keras.layers.Conv2DTranspose(256, kernel_size=5, strides=2,
                                            padding='same', use_bias=False,
                                            kernel_initializer=init),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),

            # 8×8×256 -> 16×16×128
            tf.keras.layers.Conv2DTranspose(128, kernel_size=5, strides=2,
                                            padding='same', use_bias=False,
                                            kernel_initializer=init),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),

            # 16×16×128 -> 32×32×64
            tf.keras.layers.Conv2DTranspose(64, kernel_size=5, strides=2,
                                            padding='same', use_bias=False,
                                            kernel_initializer=init),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),

            # 32×32×64 -> 64×64×3
            tf.keras.layers.Conv2DTranspose(NUM_CHANNELS, kernel_size=5, strides=2,
                                            padding='same', use_bias=False,
                                            kernel_initializer=init),
            tf.keras.layers.Activation('tanh'),                           # [-1,1]
        ],
        name='generator',
    )


# ----------------------------------------------------------------------
# Шаг 2. Проверка: G(z, training=False) → (B,64,64,3), не плоский вектор,
#         значения конечны и в [-1,1].
# ----------------------------------------------------------------------
if __name__ == '__main__':
    generator = make_generator()
    generator.summary()

    z = tf.random.normal((BATCH_SIZE, NOISE_DIM))          # z ~ N(0, I)
    x_fake = generator(z, training=False)                  # диагностический проход

    assert x_fake.shape == (BATCH_SIZE, IMAGE_HEIGHT, IMAGE_WIDTH, NUM_CHANNELS), \
        f'не та форма: {x_fake.shape}'
    x_np = x_fake.numpy()
    assert np.isfinite(x_np).all(), 'NaN/inf в G(z)'
    assert x_np.min() >= -1.0 and x_np.max() <= 1.0, \
        f'вне [-1,1]: [{x_np.min()}, {x_np.max()}]'

    print(f'z:    shape={z.shape}')
    print(f'G(z): shape={x_fake.shape}  dtype={x_fake.dtype}  '
          f'range=[{x_np.min():.3f}, {x_np.max():.3f}]')
    print(f'это не плоский (B,784), а (B,64,64,3) — да? {x_fake.shape[-1] == 3}')

## Этап 4. Свёрточный дискриминатор и логиты

**Требуется:** собрать дискриминатор из свёрток со stride 2,
LeakyReLU, Dropout и завершающего `Dense(1)` **без sigmoid**. Его
выход — логит формы $(B,1)$. Поэтому в BCE надо указать
`from_logits=True`.

**Контрольная точка:** не проверяйте логиты на принадлежность
интервалу $(0,1)$: это свойство вероятности после sigmoid, но не
самого логита.

Логит — просто оценка до перевода в вероятность. Большой положительный
логит говорит «скорее настоящее», большой отрицательный — «скорее
синтетическое». `BinaryCrossentropy(from_logits=True)` сама выполняет
нужный перевод надёжнее, чем отдельный sigmoid. D не определяет, кто
изображён на лице: он отвечает только на вопрос об источнике картинки.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import tensorflow as tf

# ----------------------------------------------------------------------
# Конфигурация (Этапы 1–3)
# ----------------------------------------------------------------------
SEED = 42
NOISE_DIM = 100
IMAGE_HEIGHT = 64
IMAGE_WIDTH = 64
NUM_CHANNELS = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-4
BETA_1 = 0.5
DROPOUT_RATE = 0.3

tf.keras.utils.set_random_seed(SEED)
artifact_dir = Path('artifacts/olivetti_faces_dense_gan')
artifact_dir.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------------
# Initializer (тот же, что у G) и фабрика G (из Этапа 3)
# ----------------------------------------------------------------------
def make_initializer() -> tf.keras.initializers.Initializer:
    return tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02, seed=SEED)


def make_generator(noise_dim: int = NOISE_DIM) -> tf.keras.Model:
    init = make_initializer()
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(noise_dim,)),
            tf.keras.layers.Dense(4 * 4 * 512, use_bias=False, kernel_initializer=init),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),
            tf.keras.layers.Reshape((4, 4, 512)),
            tf.keras.layers.Conv2DTranspose(256, 5, strides=2, padding='same',
                                            use_bias=False, kernel_initializer=init),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),
            tf.keras.layers.Conv2DTranspose(128, 5, strides=2, padding='same',
                                            use_bias=False, kernel_initializer=init),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),
            tf.keras.layers.Conv2DTranspose(64, 5, strides=2, padding='same',
                                            use_bias=False, kernel_initializer=init),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),
            tf.keras.layers.Conv2DTranspose(NUM_CHANNELS, 5, strides=2, padding='same',
                                            use_bias=False, kernel_initializer=init),
            tf.keras.layers.Activation('tanh'),
        ],
        name='generator',
    )


# ----------------------------------------------------------------------
# Шаг 1. Свёрточный D: NHWC -> Conv2D(stride=2) ×4 -> Flatten -> Dense(1)
#         БЕЗ sigmoid: выход — ЛОГИТ (B,1).
# ----------------------------------------------------------------------
def make_discriminator(dropout_rate: float = DROPOUT_RATE) -> tf.keras.Model:
    init = make_initializer()
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(IMAGE_HEIGHT, IMAGE_WIDTH, NUM_CHANNELS)),
            # 64x64x3 -> 32x32x64
            tf.keras.layers.Conv2D(64, 5, strides=2, padding='same',
                                   kernel_initializer=init),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dropout(dropout_rate),

            # 32x32x64 -> 16x16x128
            tf.keras.layers.Conv2D(128, 5, strides=2, padding='same',
                                   kernel_initializer=init),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dropout(dropout_rate),

            # 16x16x128 -> 8x8x256
            tf.keras.layers.Conv2D(256, 5, strides=2, padding='same',
                                   kernel_initializer=init),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dropout(dropout_rate),

            # 8x8x256 -> 4x4x512
            tf.keras.layers.Conv2D(512, 5, strides=2, padding='same',
                                   kernel_initializer=init),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dropout(dropout_rate),

            tf.keras.layers.Flatten(),                       # (B, 4*4*512)
            tf.keras.layers.Dense(1, kernel_initializer=init),  # ЛОГИТ (B,1)
        ],
        name='discriminator',
    )


# ----------------------------------------------------------------------
# Шаг 2. BCE с from_logits=True + ДВА независимых Adam.
#         Состояния моментов θ_G и θ_D не могут быть общими.
# ----------------------------------------------------------------------
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

generator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE, beta_1=BETA_1
)
discriminator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE, beta_1=BETA_1
)


# ----------------------------------------------------------------------
# Шаг 3. Диагностика: форма (B,1), конечность.
#         Логиты НЕ проверяем на (0,1) — это свойство вероятности, а не логита.
# ----------------------------------------------------------------------
if __name__ == '__main__':
    generator = make_generator()
    discriminator = make_discriminator()
    discriminator.summary()

    # один реальный и один синтетический пакет
    x_real = tf.random.uniform(
        (BATCH_SIZE, IMAGE_HEIGHT, IMAGE_WIDTH, NUM_CHANNELS), minval=-1.0, maxval=1.0
    )  # заглушка; на Этапе 2 сюда придёт настоящий x_real
    z = tf.random.normal((BATCH_SIZE, NOISE_DIM))
    x_fake = generator(z, training=False)

    logits_real = discriminator(x_real, training=False)
    logits_fake = discriminator(x_fake, training=False)

    for name, logits in (('D(x_real)', logits_real), ('D(G(z))', logits_fake)):
        assert logits.shape == (BATCH_SIZE, 1), (name, logits.shape)
        l_np = logits.numpy()
        assert np.isfinite(l_np).all(), f'NaN/inf in {name}'

        # ВАЖНО: проверка на (0,1) здесь НЕ делается — это логиты, а не вероятности.
        # Показываем только, что BCE(from_logits=True) их принимает.
        loss = bce(tf.ones_like(logits), logits).numpy()
        assert np.isfinite(loss), f'BCE not finite for {name}'

        print(f'{name}: shape={logits.shape}, '
              f'logit range=({l_np.min():.3f}, {l_np.max():.3f}), '
              f'BCE(with ones)={loss:.4f}')

    # демонстрация: sigmoid(logit) уже попадает в (0,1), но это НЕ выход D
    probs_demo = tf.math.sigmoid(logits_real).numpy()
    print(f'\nsigmoid(D(x_real)) range=({probs_demo.min():.4f}, {probs_demo.max():.4f}) '
          f'— только для наглядности; D возвращает ЛОГИТЫ.')

    # проверка, что оптимизаторы независимы
    assert generator_optimizer is not discriminator_optimizer
    g_vars = set(id(v) for v in generator.trainable_variables)
    d_vars = set(id(v) for v in discriminator.trainable_variables)
    assert g_vars.isdisjoint(d_vars), 'θ_G и θ_D пересекаются!'
    print(f'\nθ_G vars={len(g_vars)}, θ_D vars={len(d_vars)}, disjoint=True')
    print('BCE: from_logits=True  |  два независимых Adam (β₁=0.5)')

## Этап 5. Раздельные шаги оптимизации

**Требуется:** реализовать два `GradientTape`. В шаге дискриминатора
используйте `stop_gradient(G(z))`, реальные цели 0.9 и синтетические
0. В шаге генератора используйте новую выборку шума и целевое значение
1; градиент должен пройти через дискриминатор к изображению, но
оптимизатор меняет только переменные генератора.

**Проверка:** обе потери — конечные скаляры; после каждого подшага
изменяется только соответствующий набор параметров.

**Сначала учится D.** Настоящие лица имеют цель 0.9, рисунки — 0.
`stop_gradient` превращает рисунок в «готовую картинку» именно для
этой фазы: обратная стрелка к G выключена. **Затем учится G.** Новый
шум создаёт новый рисунок, а его цель равна 1. D остаётся полезным
проверяющим, который сообщает направление улучшения, но второй
оптимизатор получает только веса G.

`training=False` у D означает «не включать его Dropout», а не «не
передавать замечание художнику». Стрелка градиента к G сохраняется;
она исчезла бы только при `stop_gradient` в этой второй фазе.

Здесь D возвращает логит $\ell$, то есть свободную оценку до
вероятности. Мысленно вероятность равна
$p=\sigma(\ell)=1/(1+e^{-\ell})$, а ошибка одной картинки и цели $y$
записывается так:

$$
\operatorname{BCE}(y,p)=-y\log p-(1-y)\log(1-p).
$$

`BinaryCrossentropy(from_logits=True)` получает логит и объединяет
sigmoid с этой формулой устойчиво; отдельный sigmoid перед loss был бы
неверным дублированием.

### Математический и API-контракт

**Сначала прочитайте маршрут одного пакета, а уже потом заполняйте
`TODO(student)`.**

![Два независимых обновления GAN: обучение дискриминатора, затем обучение генератора](assets/diagrams/gan_two_phase_training.svg)

На рисунке синие блоки — данные, зелёные — сети с параметрами, красные
— потери-скаляры. Ваш код будет создавать временные синие тензоры,
получать два красных числа и передавать Adam только пары градиентов с
зелёными параметрами. Ни шум, ни изображения, ни классовые метки лиц
не являются объектами оптимизации.

Пусть $\theta_G$ — все веса генератора, а $\theta_D$ — все веса
дискриминатора. Потеря — одно итоговое число. Её производная по весу
отвечает, в какую сторону слегка изменить этот вес; общий шаг имеет вид

$$
\theta\leftarrow\theta-\eta\nabla_\theta\mathcal L.
$$

В первой фазе составьте $\mathcal L_D$ из BCE настоящих лиц с целью
0.9 и синтетических с целью 0, после чего берите градиент только по
$\theta_D$. Во второй составьте
$\mathcal L_G=\operatorname{BCE}(1,D(G(\mathbf z_G)))$ и берите
градиент только по $\theta_G$. Шум $\mathbf z_G$ — новый вход, а не
параметр, который оптимизатор должен сохранять.

### Как `GradientTape` помогает выполнить математическое правило

![Жизненный цикл GradientTape от прямого прохода до обновления Adam](assets/diagrams/tensorflow_gradient_tape.svg)

`GradientTape` записывает прямой путь до одной потери. Пока активен
блок `with`, TensorFlow строит временный граф дифференцируемых операций
и автоматически наблюдает задействованные обучаемые `tf.Variable`.
Вызов `tape.gradient(loss, variables)` идёт по этому графу назад и
возвращает одну производную на каждый указанный вес; её форма совпадает
с формой этого веса. Наконец, `zip` формирует пары «производная, вес»,
а `apply_gradients` передаёт их Adam. Поэтому именно список переменных
в этом вызове определяет, что обновляется.

У обычного непостоянного tape есть один обратный проход: после
`gradient(...)` его запись освобождается. Не нужно вручную строить
якобианы всех RGB-пикселей и весов; TensorFlow эффективно передаёт их
действие на градиент скалярной потери.

В фазе G путь остаётся математически равным

$$
\frac{\partial\mathcal L_G}{\partial\theta_G}
=\frac{\partial\mathcal L_G}{\partial D}
\frac{\partial D}{\partial G}
\frac{\partial G}{\partial\theta_G}.
$$

Это **цепное правило**: изменение веса G меняет картинку, картинка
меняет ответ D, а ответ D меняет итоговую потерю. D участвует в этой
цепочке как измерительный прибор, но его веса не стоят в списке
`variables`, поэтому их поправка не применяется.

### Один мини-пакет: спецификация будущего TensorFlow-кода

В самостоятельной реализации используйте следующие математические
роли и имена. Ничего из этой таблицы не надо оптимизировать, кроме
параметров $\theta_G$ и $\theta_D$.

| Объект | Имя, которое следует завести | Форма | Что проверять |
| --- | --- | --- | --- |
| Настоящие лица $\mathbf x_{\rm real}$ | `real_images` | `(B,64,64,3)` | Значения `[-1,1]`, цель D — 0.9. |
| Шум D $\mathbf z_D$ | `discriminator_noise` | `(B,100)` | Это временный вход генератора. |
| Подделка для D | `fake_images` | `(B,64,64,3)` | После `stop_gradient`; цель D — 0. |
| Логиты D | `real_logits`, `fake_logits` | `(B,1)` | Могут быть любыми вещественными числами. |
| Потеря D | `d_loss` | скаляр | Градиент берётся по `discriminator.trainable_variables`. |
| Шум G $\mathbf z_G$ | `generator_noise` | `(B,100)` | Это новая независимая выборка. |
| Изображение G | `generated_images` | `(B,64,64,3)` | Не отсоединяется от графа. |
| Потеря G | `g_loss` | скаляр | Градиент берётся по `generator.trainable_variables`. |

**Контроль перед переходом к следующей ячейке.** Разделите в уме
объекты таблицы на данные, параметры и скаляры. `fake_images` после
`stop_gradient` всё ещё имеет форму `(B,64,64,3)` и те же пиксели, но
перестаёт быть путём к $\theta_G$ только в фазе D. Во второй фазе
`generated_images` нельзя отсоединять: иначе `g_loss` не сможет дать
генератору никакого замечания.

**Порядок, который надо увидеть в своём коде.** Сначала первый tape
должен закончиться `d_loss`; следующий вызов `gradient` получает только
список весов D. Потом создаётся второй tape и новый шум; он заканчивается
`g_loss`, а `gradient` получает только список весов G. `zip` не меняет
данные — он лишь соединяет производную правильной формы с её весом.
Только `apply_gradients` делает шаг Adam и действительно изменяет сеть.

Для проверки обратного пути используйте запись

$$
\nabla_{\theta_G}\mathcal L_G=
J_{G,\theta_G}^{\mathsf T}J_{D,\mathbf x}^{\mathsf T}
\nabla_{\ell}\mathcal L_G.
$$

Якобиан — таблица локальных влияний. TensorFlow не материализует
огромные якобианы для RGB-изображений, а вычисляет их нужное действие
на градиент ошибки автоматически во время `tape.gradient`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import tensorflow as tf

# ----------------------------------------------------------------------
# Конфигурация (Этапы 1–4)
# ----------------------------------------------------------------------
SEED = 42
NOISE_DIM = 100
IMAGE_HEIGHT = 64
IMAGE_WIDTH = 64
NUM_CHANNELS = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-4
BETA_1 = 0.5
REAL_LABEL = 0.9
FAKE_LABEL = 0.0
GENERATOR_TARGET = 1.0
DROPOUT_RATE = 0.3

tf.keras.utils.set_random_seed(SEED)
artifact_dir = Path('artifacts/olivetti_faces_dense_gan')
artifact_dir.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------------
# Фабрики G и D (Этапы 3–4, без изменений)
# ----------------------------------------------------------------------
def make_initializer():
    return tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02, seed=SEED)


def make_generator(noise_dim=NOISE_DIM):
    init = make_initializer()
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(noise_dim,)),
            tf.keras.layers.Dense(4 * 4 * 512, use_bias=False, kernel_initializer=init),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),
            tf.keras.layers.Reshape((4, 4, 512)),
            tf.keras.layers.Conv2DTranspose(256, 5, 2, 'same', use_bias=False, kernel_initializer=init),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),
            tf.keras.layers.Conv2DTranspose(128, 5, 2, 'same', use_bias=False, kernel_initializer=init),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),
            tf.keras.layers.Conv2DTranspose(64, 5, 2, 'same', use_bias=False, kernel_initializer=init),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),
            tf.keras.layers.Conv2DTranspose(NUM_CHANNELS, 5, 2, 'same', use_bias=False, kernel_initializer=init),
            tf.keras.layers.Activation('tanh'),
        ],
        name='generator',
    )


def make_discriminator(dropout_rate=DROPOUT_RATE):
    init = make_initializer()
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(IMAGE_HEIGHT, IMAGE_WIDTH, NUM_CHANNELS)),
            tf.keras.layers.Conv2D(64, 5, 2, 'same', kernel_initializer=init),
            tf.keras.layers.LeakyReLU(0.2),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Conv2D(128, 5, 2, 'same', kernel_initializer=init),
            tf.keras.layers.LeakyReLU(0.2),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Conv2D(256, 5, 2, 'same', kernel_initializer=init),
            tf.keras.layers.LeakyReLU(0.2),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Conv2D(512, 5, 2, 'same', kernel_initializer=init),
            tf.keras.layers.LeakyReLU(0.2),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(1, kernel_initializer=init),   # ЛОГИТ, без sigmoid
        ],
        name='discriminator',
    )


generator = make_generator()
discriminator = make_discriminator()


# ----------------------------------------------------------------------
# BCE(from_logits=True) + ДВА независимых Adam
# ----------------------------------------------------------------------
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

generator_optimizer = tf.keras.optimizers.Adam(LEARNING_RATE, beta_1=BETA_1)
discriminator_optimizer = tf.keras.optimizers.Adam(LEARNING_RATE, beta_1=BETA_1)


# ----------------------------------------------------------------------
# train_step(real_images): (B,64,64,3) -> (L_D, L_G) — конечные скаляры.
# ----------------------------------------------------------------------
@tf.function
def train_step(real_images: tf.Tensor):
    batch_size = tf.shape(real_images)[0]

    # ================= Фаза D =================
    # z_D — свой шум; G(z_D, training=False) НЕ должен двигать θ_G.
    z_D = tf.random.normal((batch_size, NOISE_DIM))

    with tf.GradientTape() as d_tape:
        fake_images = generator(z_D, training=False)
        fake_images = tf.stop_gradient(fake_images)              # только в фазе D

        logits_real = discriminator(real_images, training=True)   # Dropout ON
        logits_fake = discriminator(fake_images, training=True)

        # цели — в форме логитов (B,1); BCE(from_logits=True) сам применит sigmoid
        d_real_loss = bce(tf.fill(tf.shape(logits_real), REAL_LABEL), logits_real)
        d_fake_loss = bce(tf.fill(tf.shape(logits_fake), FAKE_LABEL), logits_fake)
        d_loss = d_real_loss + d_fake_loss

    # градиенты ТОЛЬКО по θ_D
    d_grads = d_tape.gradient(d_loss, discriminator.trainable_variables)
    discriminator_optimizer.apply_gradients(
        zip(d_grads, discriminator.trainable_variables)
    )

    # ================= Фаза G =================
    # независимый z_G; никакого stop_gradient здесь!
    z_G = tf.random.normal((batch_size, NOISE_DIM))

    with tf.GradientTape() as g_tape:
        fake_images = generator(z_G, training=True)              # BN G в режиме обучения
        # training=False выключает Dropout D, но граф L_G -> D -> G сохраняется
        logits_fake = discriminator(fake_images, training=False)

        g_loss = bce(tf.fill(tf.shape(logits_fake), GENERATOR_TARGET), logits_fake)

    # градиенты ТОЛЬКО по θ_G
    g_grads = g_tape.gradient(g_loss, generator.trainable_variables)
    generator_optimizer.apply_gradients(
        zip(g_grads, generator.trainable_variables)
    )

    return d_loss, g_loss


# ----------------------------------------------------------------------
# Контроль
# ----------------------------------------------------------------------
if __name__ == '__main__':
    # заглушка реального пакета; на Этапе 6 сюда придёт x_real (N,64,64,3)
    real = tf.random.uniform(
        (BATCH_SIZE, IMAGE_HEIGHT, IMAGE_WIDTH, NUM_CHANNELS),
        minval=-1.0, maxval=1.0,
    )

    # 1) шаг исполняется, оба лосса конечны
    L_D, L_G = train_step(real)
    assert np.isfinite(L_D.numpy()) and np.isfinite(L_G.numpy()), (L_D, L_G)
    print(f'L_D={L_D.numpy():.4f}  L_G={L_G.numpy():.4f}')

    # 2) θ_G и θ_D не пересекаются; оптимизаторы разные
    g_vars = set(id(v) for v in generator.trainable_variables)
    d_vars = set(id(v) for v in discriminator.trainable_variables)
    assert g_vars.isdisjoint(d_vars), 'θ_G и θ_D пересекаются!'
    assert generator_optimizer is not discriminator_optimizer
    print(f'θ_G={len(g_vars)} vars, θ_D={len(d_vars)} vars, disjoint=True')

    # 3) фаза D: stop_gradient на месте -> θ_G НЕ получает градиентов
    z = tf.random.normal((BATCH_SIZE, NOISE_DIM))
    with tf.GradientTape() as tape:
        fake = tf.stop_gradient(generator(z, training=False))
        logits_fake = discriminator(fake, training=True)
        loss_d_only = bce(tf.fill(tf.shape(logits_fake), FAKE_LABEL), logits_fake)
    g_from_d = tape.gradient(loss_d_only, generator.trainable_variables)
    assert all(g is None for g in g_from_d), \
        'фаза D не должна давать градиент в θ_G — проверь stop_gradient'
    print('phase D: θ_G grads = None (stop_gradient OK)')

    # 4) фаза G: БЕЗ stop_gradient -> θ_G получает ненулевой градиент через D
    z = tf.random.normal((BATCH_SIZE, NOISE_DIM))
    with tf.GradientTape() as tape:
        fake = generator(z, training=True)                        # без stop_gradient
        logits_fake = discriminator(fake, training=False)
        loss_g_only = bce(tf.fill(tf.shape(logits_fake), GENERATOR_TARGET), logits_fake)
    g_from_g = tape.gradient(loss_g_only, generator.trainable_variables)
    n_present = sum(g is not None for g in g_from_g)
    assert n_present > 0, 'фаза G должна давать градиент в θ_G через D'
    print(f'phase G: θ_G grads present for {n_present}/{len(g_from_g)} vars '
          f'(L_G -> D -> G path OK)')

    # 5) состояния Adam независимы
    assert int(generator_optimizer.iterations.numpy()) >= 1
    assert int(discriminator_optimizer.iterations.numpy()) >= 1
    print(f'Adam steps: G={int(generator_optimizer.iterations.numpy())}, '
          f'D={int(discriminator_optimizer.iterations.numpy())}')

## Этап 6. Наблюдение генераций и выводы

**Требуется:** обучить DCGAN, сохранять сетку фиксированных шумовых
векторов через равные интервалы, построить обе истории потерь и
объяснить разнообразие либо коллапс мод. Не считайте идентификаторы
людей показателем качества GAN.

**Итоговая самопроверка:** шум `(B,100)`, изображения `(B,64,64,3)`,
логиты `(B,1)`, значения потерь конечны, метки субъектов не участвовали
в обучении.

Используйте один `fixed_noise` для всех сеток. Это одинаковые задания
для художника в начале и конце опыта; тогда видна работа обучения, а
не смена карточки. Для вывода на экран верните `[-1,1]` к `[0,1]`.
Линии потерь — полезный дневник, но не оценка красоты: смотрите на них
вместе с разнообразием лиц и отсутствием чёрных/одинаковых кадров.

In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

# ----------------------------------------------------------------------
# Конфигурация (Этапы 1–5)
# ----------------------------------------------------------------------
SEED = 42
MAX_EXAMPLES = 400
IMAGE_HEIGHT = 64
IMAGE_WIDTH = 64
NUM_CHANNELS = 3
NOISE_DIM = 100
BATCH_SIZE = 32
EPOCHS = 105                 # можно 105 / 205 / 400, меняя только это
LEARNING_RATE = 2e-4
BETA_1 = 0.5
REAL_LABEL = 0.9
FAKE_LABEL = 0.0
GENERATOR_TARGET = 1.0
SAVE_EVERY = 20
DROPOUT_RATE = 0.3
NUM_SNAPSHOT = 64            # 8×8 сетка из 64 z-векторов

tf.keras.utils.set_random_seed(SEED)
artifact_dir = Path('artifacts/olivetti_dcgan')
artifact_dir.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------------
# Данные (Этап 2)
# ----------------------------------------------------------------------
def load_dataset():
    from sklearn.datasets import fetch_olivetti_faces
    faces = fetch_olivetti_faces(shuffle=True, random_state=SEED)
    x = faces.images[:MAX_EXAMPLES].astype('float32')       # (N,64,64) [0,1]
    x = np.repeat(x[..., np.newaxis], NUM_CHANNELS, axis=-1) # (N,64,64,3)
    x = x * 2.0 - 1.0                                        # [-1,1]
    assert x.shape == (MAX_EXAMPLES, IMAGE_HEIGHT, IMAGE_WIDTH, NUM_CHANNELS)
    assert np.isfinite(x).all() and x.min() >= -1.0 and x.max() <= 1.0
    ds = (
        tf.data.Dataset.from_tensor_slices(x)
        .shuffle(len(x), seed=SEED, reshuffle_each_iteration=True)
        .batch(BATCH_SIZE, drop_remainder=True)
        .prefetch(tf.data.AUTOTUNE)
    )
    return ds, len(x) // BATCH_SIZE


# ----------------------------------------------------------------------
# Фабрики и train_step (Этапы 3–5) — вставьте сюда make_generator,
# make_discriminator, bce, два Adam и train_step из предыдущих этапов.
# ----------------------------------------------------------------------
def make_initializer():
    return tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02, seed=SEED)


def make_generator(noise_dim=NOISE_DIM):
    init = make_initializer()
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(noise_dim,)),
        tf.keras.layers.Dense(4*4*512, use_bias=False, kernel_initializer=init),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.Reshape((4, 4, 512)),
        tf.keras.layers.Conv2DTranspose(256, 5, 2, 'same', use_bias=False, kernel_initializer=init),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.Conv2DTranspose(128, 5, 2, 'same', use_bias=False, kernel_initializer=init),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.Conv2DTranspose(64, 5, 2, 'same', use_bias=False, kernel_initializer=init),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.Conv2DTranspose(NUM_CHANNELS, 5, 2, 'same', use_bias=False, kernel_initializer=init),
        tf.keras.layers.Activation('tanh'),
    ], name='generator')


def make_discriminator(dropout_rate=DROPOUT_RATE):
    init = make_initializer()
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(IMAGE_HEIGHT, IMAGE_WIDTH, NUM_CHANNELS)),
        tf.keras.layers.Conv2D(64, 5, 2, 'same', kernel_initializer=init),
        tf.keras.layers.LeakyReLU(0.2),
        tf.keras.layers.Dropout(dropout_rate),
        tf.keras.layers.Conv2D(128, 5, 2, 'same', kernel_initializer=init),
        tf.keras.layers.LeakyReLU(0.2),
        tf.keras.layers.Dropout(dropout_rate),
        tf.keras.layers.Conv2D(256, 5, 2, 'same', kernel_initializer=init),
        tf.keras.layers.LeakyReLU(0.2),
        tf.keras.layers.Dropout(dropout_rate),
        tf.keras.layers.Conv2D(512, 5, 2, 'same', kernel_initializer=init),
        tf.keras.layers.LeakyReLU(0.2),
        tf.keras.layers.Dropout(dropout_rate),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1, kernel_initializer=init),    # логиты, без sigmoid
    ], name='discriminator')


def build_train_step(generator, discriminator, g_opt, d_opt, bce):
    @tf.function
    def train_step(real_images):
        batch_size = tf.shape(real_images)[0]

        z_D = tf.random.normal((batch_size, NOISE_DIM))
        with tf.GradientTape() as d_tape:
            fake = tf.stop_gradient(generator(z_D, training=False))
            logits_real = discriminator(real_images, training=True)
            logits_fake = discriminator(fake, training=True)
            d_loss = (
                bce(tf.fill(tf.shape(logits_real), REAL_LABEL), logits_real)
                + bce(tf.fill(tf.shape(logits_fake), FAKE_LABEL), logits_fake)
            )
        d_grads = d_tape.gradient(d_loss, discriminator.trainable_variables)
        d_opt.apply_gradients(zip(d_grads, discriminator.trainable_variables))

        z_G = tf.random.normal((batch_size, NOISE_DIM))
        with tf.GradientTape() as g_tape:
            fake = generator(z_G, training=True)
            logits_fake = discriminator(fake, training=False)
            g_loss = bce(tf.fill(tf.shape(logits_fake), GENERATOR_TARGET), logits_fake)
        g_grads = g_tape.gradient(g_loss, generator.trainable_variables)
        g_opt.apply_gradients(zip(g_grads, generator.trainable_variables))

        return d_loss, g_loss
    return train_step


# ----------------------------------------------------------------------
# Снимок сетки: G(fixed_noise, training=False) -> [-1,1] -> [0,1] -> imshow
# ----------------------------------------------------------------------
def save_grid(generator, fixed_noise, path, title, n=NUM_SNAPSHOT):
    fake = generator(fixed_noise[:n], training=False)         # (n,64,64,3) в [-1,1]
    imgs = (fake.numpy() + 1.0) / 2.0                         # в [0,1]
    imgs = np.clip(imgs, 0.0, 1.0)

    side = int(np.ceil(np.sqrt(n)))
    fig, axes = plt.subplots(side, side, figsize=(side*1.4, side*1.4))
    for i, ax in enumerate(axes.flat):
        if i < n:
            ax.imshow(imgs[i], vmin=0, vmax=1)                # явный диапазон
        ax.axis('off')
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    fig.savefig(path, dpi=80)
    plt.close(fig)


def plot_losses(history, path, title):
    ep = np.arange(1, len(history['d_loss']) + 1)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(ep, history['d_loss'], label='L_D')
    ax.plot(ep, history['g_loss'], label='L_G')
    ax.set_xlabel('epoch'); ax.set_ylabel('loss')
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


# ----------------------------------------------------------------------
# Один прогон: меняется ТОЛЬКО EPOCHS
# ----------------------------------------------------------------------
def run_experiment(n_epochs: int):
    tf.keras.utils.set_random_seed(SEED)                      # сброс θ_G, θ_D, шумов

    ds, steps_per_epoch = load_dataset()
    generator = make_generator()
    discriminator = make_discriminator()

    bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)
    g_opt = tf.keras.optimizers.Adam(LEARNING_RATE, beta_1=BETA_1)
    d_opt = tf.keras.optimizers.Adam(LEARNING_RATE, beta_1=BETA_1)
    train_step = build_train_step(generator, discriminator, g_opt, d_opt, bce)

    # ОДИН fixed_noise на весь прогон (и одинаковый между прогонами).
    fixed_noise = tf.random.normal((NUM_SNAPSHOT, NOISE_DIM))

    run_dir = artifact_dir / f'epochs_{n_epochs}'
    run_dir.mkdir(parents=True, exist_ok=True)

    history = {'d_loss': [], 'g_loss': []}
    t0 = time.time()
    for epoch in range(1, n_epochs + 1):
        d_vals, g_vals = [], []
        for real_batch in ds:
            d_loss, g_loss = train_step(real_batch)
            # скаляры -> числа строго после выхода из tf.function
            d_vals.append(float(d_loss.numpy()))
            g_vals.append(float(g_loss.numpy()))

        history['d_loss'].append(float(np.mean(d_vals)))
        history['g_loss'].append(float(np.mean(g_vals)))

        if epoch == 1 or epoch % SAVE_EVERY == 0:
            save_grid(generator, fixed_noise,
                      run_dir / f'samples_epoch_{epoch:03d}.png',
                      f'{n_epochs} ep — epoch {epoch:3d}  '
                      f'L_D={history["d_loss"][-1]:.3f}  L_G={history["g_loss"][-1]:.3f}')
            print(f'[{n_epochs}] ep {epoch:3d}: '
                  f'L_D={history["d_loss"][-1]:.4f}  L_G={history["g_loss"][-1]:.4f}')

    elapsed = time.time() - t0
    plot_losses(history, run_dir / 'losses.png', f'DCGAN losses — {n_epochs} epochs')
    save_grid(generator, fixed_noise, run_dir / 'samples_final.png',
              f'final after {n_epochs} epochs')
    (run_dir / 'history.json').write_text(json.dumps(history))

    return {
        'epochs': n_epochs,
        'steps_per_epoch': steps_per_epoch,
        'final_d_loss': history['d_loss'][-1],
        'final_g_loss': history['g_loss'][-1],
        'seconds': round(elapsed, 1),
        'run_dir': str(run_dir),
    }


# ----------------------------------------------------------------------
# Сравнение 105 / 205 / 400: меняется только EPOCHS
# ----------------------------------------------------------------------
if __name__ == '__main__':
    EPOCH_SCHEDULE = (105, 205, 400)
    results = [run_experiment(n) for n in EPOCH_SCHEDULE]

    (artifact_dir / 'comparison.json').write_text(json.dumps(results, indent=2))

    fig, ax = plt.subplots(figsize=(7, 4))
    xs = [r['epochs'] for r in results]
    ax.plot(xs, [r['final_d_loss'] for r in results], 'o-', label='final L_D')
    ax.plot(xs, [r['final_g_loss'] for r in results], 's-', label='final L_G')
    ax.set_xlabel('epochs'); ax.set_ylabel('final loss')
    ax.set_title('105 vs 205 vs 400 epochs (остальное фиксировано)')
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(artifact_dir / 'comparison.png', dpi=100)
    plt.close(fig)

    for r in results:
        print(f"{r['epochs']:3d} ep | steps/ep={r['steps_per_epoch']} "
              f"| final L_D={r['final_d_loss']:.4f} L_G={r['final_g_loss']:.4f} "
              f"| {r['seconds']}s")